# 01 - Combine Seasons
Combines 12 seasons of Premier League match data (2014/15–2025/26) from football-data.co.uk into one clean dataset for feature engineering.

**Input:** `data/raw/*.csv` 

**Output:** `data/processed/matches.csv`

## Imports and file paths
Import libraries and define the input and output paths.

In [1]:
import pandas as pd
from pathlib import Path

raw_dir = Path("../data/raw")
processed_path = Path("../data/processed/matches.csv")
raw_files = sorted(raw_dir.glob("*.csv"))

## Load and label each season
Read each season’s CSV, drop the blank trailing rows that some files contain, and add a `Season` column. Seasons are labelled by their starting year (2014 = 2014/15), so they can be sorted and compared numerically.


In [2]:
all_seasons = []
for file in raw_files:
    season_df = pd.read_csv(file)
    season_df = season_df.dropna(subset=["HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"])
    season_year = int(file.stem.split("_")[2])
    season_col = pd.DataFrame({"Season": season_year}, index=season_df.index)
    season_df = pd.concat([season_col, season_df], axis=1)
    all_seasons.append(season_df)
print(f"Seasons Loaded: {[int(season['Season'].unique()[0]) for season in all_seasons]}")   


Seasons Loaded: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


## Combine into one table
Stack all 12 seasons into a single DataFrame of 4,560 matches.

In [3]:
df = pd.concat(all_seasons, ignore_index=True)
print(f"Combined dataframe shape: {df.shape}, Seasons: {df['Season'].unique()}")

Combined dataframe shape: (4560, 184), Seasons: [2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025]


## Validate
Every season should contain exactly 380 matches (20 teams × 38 games ÷ 2), and the core match columns should have no missing values.

In [4]:
assert (df["Season"].value_counts() == 380).all(), "Every season should have 380 matches"
assert df[["Season", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Date", "HS", "AS", "HST", "AST"]].isna().sum().sum() == 0, "There should be no missing values in the essential columns"

## Find columns shared by every season
Column sets vary between seasons, as bookmakers are added and dropped and naming conventions change. Intersecting them shows which columns can be relied on across the full dataset.

In [5]:
common_columns = set(all_seasons[0].columns)
for season_df in all_seasons[1:]:
    common_columns = common_columns.intersection(set(season_df.columns))
common_columns = sorted(common_columns)
print(f"Common columns across all seasons: {common_columns}")

Common columns across all seasons: ['AC', 'AF', 'AR', 'AS', 'AST', 'AY', 'AwayTeam', 'B365A', 'B365D', 'B365H', 'BWA', 'BWD', 'BWH', 'Date', 'Div', 'FTAG', 'FTHG', 'FTR', 'HC', 'HF', 'HR', 'HS', 'HST', 'HTAG', 'HTHG', 'HTR', 'HY', 'HomeTeam', 'PSA', 'PSCA', 'PSCD', 'PSCH', 'PSD', 'PSH', 'Referee', 'Season']


## Choose a bookmaker benchmark
Bookmaker odds serve as the benchmark the model must compete with, so they need to be complete, especially in the 2024/25 and 2025/26 test seasons.

In [6]:
missing_pinnacle = df["PSH"].isna()
rows_missing_pinnacle = df[missing_pinnacle]
print(f"Matches missing Pinnacle odds: {rows_missing_pinnacle.shape[0]} (Seasons: {rows_missing_pinnacle['Season'].unique()})")
assert df[["B365H", "B365D", "B365A"]].isna().sum().sum() == 0, "There should be no missing values in the B365 columns"
print(f"Bet365 odds complete for all {len(df)} matches.")

Matches missing Pinnacle odds: 170 (Seasons: [2025])
Bet365 odds complete for all 4560 matches.


**Note:** Pinnacle is the sharpest benchmark, but its public API closed in July 2025, leaving 170 matches in 2025/26 without odds. Bet365 is complete for every match, so it’s used as the benchmark.

## Select columns
Keep identifiers, the target (`FTR`), match statistics, and Bet365 odds. Match statistics like shots are only known after kickoff, so they’re used only to build features from previous matches, never as direct inputs.

In [7]:
keep_cols = [
    "Season", "Date", "HomeTeam", "AwayTeam", # Match Identifiers
    "FTHG", "FTAG", "FTR", # Goals and Result
    "HS", "AS", "HST", "AST", # Shots and Shots on Target
    "B365H", "B365D", "B365A", # Bet365 Odds
    "HC", "AC", "HY", "AY", "HR", "AR" # Cards and Corners
]
matches = df[keep_cols].copy()
print(f"Final matches dataframe shape: {matches.shape}, Seasons: {matches['Season'].unique()}")


Final matches dataframe shape: (4560, 20), Seasons: [2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025]


## Parse dates
Dates are stored day-first, and older seasons use 2-digit years, so they’re parsed with `dayfirst=True` and `format="mixed"`. Each season’s first and last date confirms the parsing.

In [8]:
matches["Date"] = pd.to_datetime(matches["Date"], dayfirst=True, format="mixed")
assert matches["Date"].isna().sum() == 0, "There should be no missing dates"
matches.groupby("Season")["Date"].agg(["min", "max"])



,min,max
Season,,
2014,2014-08-16,2015-05-24
2015,2015-08-08,2016-05-17
2016,2016-08-13,2017-05-21
2017,2017-08-11,2018-05-13
2018,2018-08-10,2019-05-12
2019,2019-08-09,2020-07-26
2020,2020-09-12,2021-05-23
2021,2021-08-13,2022-05-22
2022,2022-08-05,2023-05-28


**Note:** 2019/20 ends in July 2020 and 2020/21 starts in September 2020 because of the COVID-19 suspension. These dates are correct.

## Check team names
Confirm team names are spelled consistently and that the same teams appear in the home and away columns.

In [9]:
home_teams = sorted(matches["HomeTeam"].unique())
away_teams = sorted(matches["AwayTeam"].unique())
assert home_teams == away_teams, "Home and Away teams should be the same across all seasons"
print(f"{len(home_teams)} Teams across all seasons: {home_teams}")

35 Teams across all seasons: ['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Cardiff', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Huddersfield', 'Hull', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Middlesbrough', 'Newcastle', 'Norwich', "Nott'm Forest", 'QPR', 'Sheffield United', 'Southampton', 'Stoke', 'Sunderland', 'Swansea', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wolves']


## Sort and save
Sort matches chronologically, since later feature engineering depends on time order, and save to `data/processed/`. CSVs don’t store data types, so reload this file with `parse_dates=["Date"]`.

In [10]:
matches = matches.sort_values("Date").reset_index(drop=True)
matches.to_csv(processed_path, index=False)
check = pd.read_csv(processed_path, parse_dates=["Date"])
assert check.shape == matches.shape, "Saved and loaded dataframe shapes should match"
assert pd.api.types.is_datetime64_any_dtype(check["Date"]), "Date column should be datetime type after loading"
print(f"Processed matches saved to {processed_path}, shape: {check.shape}, Seasons: {check['Season'].unique()}")

Processed matches saved to ../data/processed/matches.csv, shape: (4560, 20), Seasons: [2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025]


## Summary
- 4,560 matches across 12 seasons, 20 columns, no missing values
- Bet365 odds used as the benchmark (Pinnacle is incomplete for 2025/26)
- COVID-affected dates verified as real
- 2024/25 and 2025/26 are reserved as the held-out test set